# Re-training the 5 runs that are far from their papers (DDECC × 2, SB-ECC × 2, CrossMPT × 1)

Each run is trained again **with its official code and official settings** (a fresh clone of each official repository) and a **new seed**, then tested by the papers' rule (≥ 500 frame errors).

| run | code | earlier reproduction − paper (4 / 5 / 6 dB) |
|---|---|---|
| DDECC | BCH(63,36) | −0.41 / −0.70 / −0.96 (LS) |
| DDECC | POLAR(128,86) | −0.41 / −0.53 / −1.02 (LS) |
| SB-ECC | POLAR(128,64) | −0.97 / −1.45 / −0.96 |
| SB-ECC | CCSDS(128,64) | −1.03 / −1.74 / −2.48 |
| CrossMPT | POLAR(128,64) | −1.93 / −2.49 / −1.76 |

DDECC is trained **once** per code and tested twice, with regular (reg) and with line-search (LS) reverse diffusion: both decode with the same model, as in DDECC's own `Main.py`.

**Steps** (after a disconnect: run step 1, then the step you were in; training and testing both continue where they stopped)

| step | what | GPU? | about |
|---|---|---|---|
| 1 | setup: Drive, settings, helper functions | no | |
| 2 | environment: clone + patch the 3 official repos | yes | 1 min |
| 3 | check: what is trained / tested | no | |
| 4 | train the 5 runs | yes | ~38 h in total |
| 5 | test them (≥ 500 frame errors or 1e7 frames) | yes | ~1–2 days in total |
| 6 | results: new vs. paper vs. earlier reproduction, figures | no | |

Everything goes to `MyDrive/DDECC_results/retrain_v4/`: `repos/` (the 3 official repos), `runs/` (training), `tests/`, `figures/`. No earlier result is touched.

## 1. Setup (no GPU)

**Official settings of each method** (from each repository's `Main` script, README and paper):

| | DDECC | SB-ECC | CrossMPT |
|---|---|---|---|
| repo | yoniLc/DDECC | alonhelvits/SB-ECC | iil-postech/crossmpt |
| N / d / heads | 6 / 128 / 8 | 6 / 128 / 8 | 6 / 128 / 8 |
| lr (cosine to) | 1e-4 (5e-6), paper | 5e-4 (5e-6), `train.sh` | 1e-4 (1e-6), script default |
| epochs × batches | 2000 × 1000 | 1500 × 1000 | 1000 × 1000 |
| batch | 128 | 256 if n ≤ 64, else 128 (paper) | 128 |
| other | β = 0.01 | σ 0.1–0.8, Euler solver, 10 steps | `--standardize` for BCH only (README) |
| model that is tested | `best_model` (lowest loss) | `best_model` (lowest loss) | model after the last epoch (the script tests that one) |

**Seed.** The earlier runs used each script's default seed 42. Training again with 42 would mostly repeat the same run, so the default here is seed 1 (`SEED`). Change it to try other seeds; every seed gets its own folder.

**Test.** Random codewords + AWGN, decoded with each method's own decoder, counted like each method's own `test()`. Stop at > 500 frame errors and > 1e5 frames, or at `MAX_FRAMES` frames. The official caps are 1e7 (DDECC), 1e8 (SB-ECC) and 1e9 (CrossMPT). The default here is 1e7 for all three, so a 6 dB point takes hours, not days. Set `MAX_FRAMES_BY_METHOD` to the official values if you want them.

In [ ]:
import os, sys, re, json, math, time, subprocess
from pathlib import Path
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# ============================== settings ==============================
OUT = Path("/content/drive/MyDrive/DDECC_results/retrain_v4")
SEED = 1
JOBS = [("DDECC", ("BCH", 63, 36)),
        ("DDECC", ("POLAR", 128, 86)),
        ("SB-ECC", ("POLAR", 128, 64)),
        ("SB-ECC", ("CCSDS", 128, 64)),
        ("CrossMPT", ("POLAR", 128, 64))]
EBNO = [4, 5, 6]
MIN_FER, MIN_FRAMES = 500, 1e5
MAX_FRAMES_BY_METHOD = {"DDECC": 1e7, "SB-ECC": 1e7, "CrossMPT": 1e7}   # official: 1e7 / 1e8 / 1e9
TEST_BATCH, SAVE_EVERY, SAVE_EVERY_S = 2048, 50, 120
WORKERS = min(4, os.cpu_count() or 1)
# ======================================================================

REPOS = OUT / "repos"          # the 3 official repos, cloned to Drive (kept across sessions)
RUNS, TESTS, FIGS = OUT / "runs", OUT / "tests", OUT / "figures"
for p in (RUNS, TESTS, FIGS):
    p.mkdir(parents=True, exist_ok=True)

METHODS = {
    "DDECC": dict(url="https://github.com/yoniLc/DDECC.git", script="Main.py", test_model="best_model",
                  modes=["reg", "LS"], epochs=2000),
    "SB-ECC": dict(url="https://github.com/alonhelvits/SB-ECC.git", script="Main.py", test_model="best_model",
                   modes=["euler"], epochs=1500),
    "CrossMPT": dict(url="https://github.com/iil-postech/crossmpt.git", script="Main_CrossMPT.py", test_model="last_model",
                     modes=["direct"], epochs=1000),
}


def train_args(method, c):
    t, n, k = c
    a = ["--gpus=0", f"--code_type={t}", f"--code_n={n}", f"--code_k={k}", "--N_dec=6", "--d_model=128", "--h=8",
         f"--test_batch_size={TEST_BATCH}", f"--workers={WORKERS}", f"--seed={SEED}", f"--epochs={METHODS[method]['epochs']}"]
    if method == "DDECC":
        a += ["--lr=1e-4", "--batch_size=128", "--sigma=0.01"]
    elif method == "SB-ECC":
        a += ["--lr=5e-4", f"--batch_size={256 if n <= 64 else 128}", "--solver=euler", "--sigma_min=0.1", "--sigma_max=0.8"]
    else:
        a += ["--lr=1e-4", "--batch_size=128"] + (["--standardize"] if t == "BCH" else [])
    return a


def name(c):
    return f"{c[0]}({c[1]},{c[2]})"


def label(method, c):
    return f"{method}_{c[0]}_n{c[1]}_k{c[2]}_seed{SEED}"


def read(p):
    try:
        return Path(p).read_text(errors="ignore")
    except OSError:
        return ""


def read_int(p):
    try:
        return int(Path(p).read_text().strip())
    except (OSError, ValueError):
        return None


def save_json(path, obj):
    for _ in range(10):                      # Drive can fail for a moment: retry
        try:
            tmp = Path(str(path) + ".tmp")
            tmp.write_text(json.dumps(obj, indent=1))
            os.replace(tmp, path)
            return
        except OSError as e:
            print("save failed, retrying:", e, flush=True)
            time.sleep(10)


def train_state(method, c):
    d = RUNS / label(method, c)
    E = METHODS[method]["epochs"]
    last = read_int(d / "last_epoch.txt") or 0
    model = d / METHODS[method]["test_model"]
    if (d / "train_done.txt").exists() and model.exists():
        return dict(state="trained", dir=d, model=str(model), epoch=E)
    if last and (d / "last_model").exists():
        return dict(state="resume", dir=d, epoch=last)
    return dict(state="fresh", dir=d, epoch=0)


def test_file(method, c, mode):
    return TESTS / f"{label(method, c)}_{mode}.json"


def test_result(method, c, mode, e):
    s = json.loads(read(test_file(method, c, mode)) or "{}").get(str(e))
    if not s or not s.get("frames"):
        return None
    n = c[1]
    return dict(ber=s["be"] / (s["frames"] * n), fer=s["fe"] / s["frames"], frames=s["frames"], fe=s["fe"], done=s["done"])


def lnb(r):
    return -math.log(r["ber"]) if r and r["ber"] > 0 else float("inf")


def check(verbose=True):
    S = {}
    for method, c in JOBS:
        tr = train_state(method, c)
        res = {(m, e): test_result(method, c, m, e) for m in METHODS[method]["modes"] for e in EBNO}
        S[(method, c)] = dict(train=tr, res=res)
    if verbose:
        print(f"{'run':<26}{'training':<26}{'tests (mode: 4/5/6 dB)'}")
        print("-" * 100)
        for (method, c), s in S.items():
            tr = s["train"]; E = METHODS[method]["epochs"]
            st = {"trained": f"✅ trained {E}/{E}", "resume": f"⏸ epoch {tr['epoch']}/{E}", "fresh": "⬜ not started"}[tr["state"]]
            tests = "   ".join(f"{m}: " + " ".join("✓" if r and r["done"] else "…" if r else "—"
                                                    for r in (s["res"][(m, e)] for e in EBNO))
                               for m in METHODS[method]["modes"])
            print(f"{method + ' ' + name(c):<26}{st:<26}{tests}")
        print("-" * 100)
        print("✓ finished, … running / stopped (continues), — not started")
    return S


# ---------------------------- official repos: patches + test program ----------------------------
HELPERS = """def _safe_save(obj, path, retries=10, wait=30):
    for i in range(retries):            # temp file + rename; retry on short Drive outages
        try:
            torch.save(obj, path + '.tmp')
            os.replace(path + '.tmp', path)
            return True
        except Exception as e:
            print(f'[save] {path} failed ({e}); retry {i + 1}/{retries}', flush=True)
            time.sleep(wait)
    return False


def _safe_write(path, text, retries=10, wait=30):
    for i in range(retries):
        try:
            with open(path + '.tmp', 'w') as f:
                f.write(text)
            os.replace(path + '.tmp', path)
            return True
        except Exception as e:
            print(f'[write] {path} failed ({e}); retry {i + 1}/{retries}', flush=True)
            time.sleep(wait)
    return False


def main(args):"""

PER_REPO = {
    "DDECC": dict(model="    model = DDECCT(args, device=device,dropout=0).to(device)\n    model.ema.register(model)\n",
                  eta="eta_min=5e-6)\n",
                  midtest="        if epoch % (args.epochs//2) == 0 or epoch in [1,25]:\n",
                  final="    logging.info('Loading Best Model')\n"),
    "SB-ECC": dict(model="    model = SB_ECC(args, device=device, dropout=0).to(device)\n    model.ema.register(model)\n",
                   eta="eta_min=5e-6)\n",
                   midtest="        if (args.epochs > 1 and epoch % (args.epochs // 2) == 0) or epoch in [5, 15, 25, 50, 75, 100]:\n",
                   final="    logging.info('Loading Best Model')\n"),
    "CrossMPT": dict(model="    model = ECC_Transformer(args, dropout=0).to(device)\n",
                     eta="eta_min=1e-6)\n",
                     midtest="        if epoch % 300 == 0 or epoch in [1,args.epochs]:\n",
                     final=None),
}


def rep(src, old, new, what):
    k = src.count(old)
    assert k == 1, f"patch '{what}' failed: the original text occurs {k} times (expected 1)"
    return src.replace(old, new)


def patch(method):
    d = REPOS / method
    if not (d / ".git").exists():
        subprocess.run(["git", "clone", "-q", METHODS[method]["url"], str(d)], check=True)
    script = METHODS[method]["script"]
    subprocess.run(["git", "-C", str(d), "checkout", "--", script], check=True)
    P = PER_REPO[method]
    src = (d / script).read_text()
    src = rep(src, "def main(args):", HELPERS, "helpers")
    src = rep(src, "    parser.add_argument('--seed', type=int, default=42)\n",
              "    parser.add_argument('--seed', type=int, default=42)\n"
              "    parser.add_argument('--out_dir', type=str, default='')\n"
              "    parser.add_argument('--resume', type=str, default='')\n"
              "    parser.add_argument('--start_epoch', type=int, default=1)\n"
              "    parser.add_argument('--save_every', type=int, default=50)\n"
              "    parser.add_argument('--no_test', action='store_true')\n", "arguments")
    src = rep(src, "    os.makedirs(model_dir, exist_ok=True)\n",
              "    if args.out_dir:\n        model_dir = args.out_dir\n    os.makedirs(model_dir, exist_ok=True)\n", "out_dir")
    orig = "".join("    " + l + "\n" for l in P["model"].rstrip("\n").split("\n"))
    src = rep(src, P["model"],
              "    if args.resume:\n"
              "        model = torch.load(args.resume, map_location=device, weights_only=False).to(device)\n"
              "        if hasattr(model, 'device'):\n"
              "            model.device = device\n"
              "        logging.info(f'Resumed from {args.resume}, start_epoch={args.start_epoch}')\n"
              "    else:\n" + orig, "model creation")
    line = [l for l in src.split("\n") if "CosineAnnealingLR(optimizer" in l and P["eta"].rstrip("\n") in l]
    assert len(line) == 1, "scheduler line not found"
    src = rep(src, line[0] + "\n", line[0] + "\n"
              "    for _ in range(args.start_epoch - 1):\n"
              "        scheduler.step()\n"
              "    _opt = os.path.join(os.path.dirname(args.resume), 'last_optim.pt') if args.resume else ''\n"
              "    if _opt and os.path.exists(_opt):\n"
              "        _cur = optimizer.state_dict()\n"
              "        _cur['state'] = torch.load(_opt, map_location=device, weights_only=False)['state']\n"
              "        optimizer.load_state_dict(_cur)\n"
              "        logging.info('Adam state restored')\n", "scheduler")
    src = rep(src, "    best_loss = float('inf')\n    for epoch in range(1, args.epochs + 1):\n",
              "    best_loss = float('inf')\n"
              "    _bl = os.path.join(os.path.dirname(args.resume), 'best_loss.txt') if args.resume else ''\n"
              "    if _bl and os.path.exists(_bl):\n"
              "        best_loss = float(open(_bl).read())\n"
              "    for epoch in range(args.start_epoch, args.epochs + 1):\n", "epoch loop")
    src = rep(src, "            torch.save(model, os.path.join(args.path, 'best_model'))\n",
              "            if _safe_save(model, os.path.join(args.path, 'best_model')):\n"
              "                _safe_write(os.path.join(args.path, 'best_loss.txt'), repr(best_loss))\n", "best save")
    # periodic last_model: right before the (patched) test condition, i.e. inside the epoch loop
    src = rep(src, P["midtest"],
              "        if epoch % args.save_every == 0 or epoch == args.epochs:\n"
              "            if (_safe_save(model, os.path.join(args.path, 'last_model')) and\n"
              "                    _safe_save(optimizer.state_dict(), os.path.join(args.path, 'last_optim.pt'))):\n"
              "                _safe_write(os.path.join(args.path, 'last_epoch.txt'), str(epoch))\n"
              + P["midtest"].replace("        if ", "        if (not args.no_test) and (", 1).replace(":\n", "):\n"), "test switch")
    if P["final"]:
        src = rep(src, P["final"], "    if args.no_test:\n        logging.info('Training finished (testing is done by the notebook)')\n"
                  "        return\n" + P["final"], "final test")
    compile(src, script, "exec")
    (d / script).write_text(src)


EVAL = r"""
# Resumable test of one trained model with one decoding (written by the notebook, run by step 5)
import argparse, json, math, os, sys, time
ap = argparse.ArgumentParser()
for a in ("method", "repo", "code_type", "model", "state", "mode"):
    ap.add_argument("--" + a, required=True)
for a in ("n", "k", "seed", "batch"):
    ap.add_argument("--" + a, type=int, required=True)
for a in ("min_fer", "min_frames", "max_frames", "save_every_s"):
    ap.add_argument("--" + a, type=float, required=True)
ap.add_argument("--ebno", type=str, required=True)
a = ap.parse_args()
sys.path.insert(0, a.repo); os.chdir(a.repo)
import torch
from Codes import Get_Generator_and_Parity, EbN0_to_std, sign_to_bin, bin_to_sign


class Code:                                   # the saved models refer to __main__.Code
    pass


dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
code = Code(); code.n, code.k, code.code_type = a.n, a.k, a.code_type
if a.method == "CrossMPT":                    # CrossMPT: systematic H only for BCH (--standardize)
    G, H = Get_Generator_and_Parity(code, standard_form=(a.code_type == "BCH"))
else:
    G, H = Get_Generator_and_Parity(code)
G = torch.from_numpy(G).long(); H = torch.from_numpy(H).long()
model = torch.load(a.model, map_location=dev, weights_only=False)
if hasattr(model, "device"):
    model.device = dev
model.eval()

if a.method == "DDECC":
    import DDECC

    def p_sample(self, yt):
        # DDECC.py's p_sample (regular / line search); indices on the CPU, line search on the GPU
        yt = yt.cpu()
        ss = (torch.matmul(sign_to_bin(torch.sign(yt.to(self.device))), self.pc_matrix) % 2).round().long().sum(-1)
        t = ss.cpu()
        nm = self(yt.to(self.device), ss.to(self.device)).cpu()
        na = yt - torch.sign(-nm * torch.sign(yt))
        bb, b = self.betas_bar.cpu(), self.betas.cpu()
        factor = torch.sqrt(bb[t]) * b[t] / (bb[t] + b[t])
        alpha_final = 1
        if self.line_search:
            alpha = torch.linspace(1, 20, 20, device=self.device).view(1, 1, -1)
            step = (na * factor).to(self.device).unsqueeze(-1)
            cand = sign_to_bin(torch.sign(yt.to(self.device).unsqueeze(-1) - alpha * step)).permute(0, 2, 1)
            new_synd = (torch.matmul(cand, self.pc_matrix) % 2).round().long().sum(-1)
            alpha_final = alpha.view(-1)[new_synd.argmin(-1)].cpu().unsqueeze(-1)
        yt_1 = yt - alpha_final * na * factor
        yt_1[t == 0] = yt[t == 0]
        return yt_1, t

    DDECC.DDECCT.p_sample = p_sample
    model.line_search = (a.mode == "LS")

    def decode(x, y):
        x_pred, _, _, _ = model.p_sample_loop(y)
        return sign_to_bin(torch.sign(x_pred)).cpu()
elif a.method == "SB-ECC":
    def decode(x, y):
        x_pred, _, _, _ = model.p_sample_loop(y, solver_type="euler")
        return sign_to_bin(torch.sign(x_pred)).cpu()
else:
    def decode(x, y):                         # as test() in Main_CrossMPT.py
        magnitude = torch.abs(y)
        syndrome = bin_to_sign(torch.matmul(sign_to_bin(torch.sign(y)).long(), H.t()) % 2).float()
        z_pred = model(magnitude.to(dev), syndrome.to(dev))
        _, x_pred = model.loss(-z_pred, (y * bin_to_sign(x)).to(dev), y.to(dev))
        return x_pred.cpu()

meta = dict(method=a.method, model=a.model, mode=a.mode, min_fer=a.min_fer, min_frames=a.min_frames,
            max_frames=a.max_frames, batch=a.batch, seed=a.seed)
state = json.load(open(a.state)) if os.path.exists(a.state) else {"meta": meta}
if state.get("meta") != meta:
    os.rename(a.state, a.state + f".bak{int(time.time())}"); state = {"meta": meta}
    print("state made with other settings -> renamed to .bak, starting over", flush=True)


def save():
    for _ in range(10):
        try:
            json.dump(state, open(a.state + ".tmp", "w"), indent=1); os.replace(a.state + ".tmp", a.state); return
        except OSError as e:
            print("save failed, retrying:", e, flush=True); time.sleep(10)


def log(m):
    print(time.strftime("[%m-%d %H:%M:%S] ") + m, flush=True)


for ebno in [int(e) for e in a.ebno.split(",")]:
    st = state.setdefault(str(ebno), dict(frames=0, batches=0, be=0, fe=0, done=False, seconds=0.0))
    if st["done"]:
        log(f"[{a.mode}] {ebno} dB: already finished"); continue
    if st["frames"]:
        log(f"[{a.mode}] {ebno} dB: resuming at {st['frames']:.3e} frames, {st['fe']} frame errors")
    std = EbN0_to_std(ebno, a.k / a.n)
    t0 = last = start = time.time(); fr0 = st["frames"]
    try:
        while True:
            gen = torch.Generator().manual_seed(a.seed * 1000003 + ebno * 10007 + st["batches"])
            x = (torch.randint(0, 2, (a.batch, a.k), generator=gen) @ G) % 2
            y = (bin_to_sign(x) + torch.randn(a.batch, a.n, generator=gen) * std).float()
            with torch.no_grad():
                xh = decode(x, y)
            err = xh != x.float()
            st["be"] += int(err.sum()); st["fe"] += int(err.any(dim=1).sum())
            st["frames"] += a.batch; st["batches"] += 1
            fin = (st["fe"] > a.min_fer and st["frames"] > a.min_frames) or st["frames"] >= a.max_frames
            if fin or time.time() - last > a.save_every_s:
                st["seconds"] += time.time() - t0; t0 = last = time.time(); st["done"] = fin; save()
                rate = (st["frames"] - fr0) / max(time.time() - start, 1e-9)
                l = f"{-math.log(st['be'] / (st['frames'] * a.n)):.3f}" if st["be"] else "inf"
                log(f"  [{a.mode}] EbN0={ebno}: {st['frames']:.3e} frames, {st['fe']} frame errors, {st['be']} bit errors, "
                    f"-ln(BER)={l}, {rate:.0f} frames/s, " + ("done" if fin else f"at most {(a.max_frames - st['frames']) / max(rate, 1e-9) / 3600:.1f} h left"))
            if fin:
                break
    except KeyboardInterrupt:
        st["seconds"] += time.time() - t0; save(); log("stopped by hand, progress saved"); raise
"""


def ensure_env():
    REPOS.mkdir(parents=True, exist_ok=True)
    for m in sorted({m for m, _ in JOBS}):
        patch(m)
    Path("/content/eval_job.py").write_text(EVAL)



SETUP_OK = True
print("Setup done. Output folder:", OUT)

## 2. Environment: clone + patch the 3 official repos (GPU)

Each repository is cloned once into its own Drive folder, `retrain_v4/repos/<method>/` (kept across sessions: a later run only checks it is there). Before patching, every run restores the original script (`git checkout`), so patches are never applied twice. Each patch checks that the text it replaces occurs exactly once.

**What the patches change.** They only add saving and resuming; model, loss, data and schedule are untouched:
- `--out_dir`: a fixed run folder on Drive.
- `--resume` / `--start_epoch`: continue from `last_model`. The cosine lr is fast-forwarded, and the Adam state (`last_optim.pt`) and `best_loss` are restored, so a resumed run trains like an uninterrupted one.
- `last_model` + `last_epoch.txt` every 50 epochs and at the last epoch. Saves are safe: temp file + rename, retried on Drive outages.
- `--no_test`: the scripts' own tests are skipped. Step 5 tests instead, with ≥ 500 frame errors and resumable.

This cell also writes the test program `/content/eval_job.py`, which step 5 runs once per run and decoding.

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

ensure_env()
import torch
print("Repos cloned and patched:", ", ".join(sorted({m for m, _ in JOBS})), "| test program: /content/eval_job.py")
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> Runtime → Change runtime type → GPU")

## 3. Check (no GPU)

For each run: training progress, and per decoding and Eb/N0 whether the test is finished (✓), running or stopped (…, continues in step 5) or not started (—).

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

S = check()

## 4. Train the 5 runs (GPU)

Runs the patched official scripts one after the other, in the order of `JOBS`. Each run goes to `retrain_v4/runs/<method>_<code>_seed<SEED>/` (`logging.txt`, `best_model`, `last_model`, …).

Rough GPU time on Colab, from your earlier logs:
- DDECC BCH(63,36): about 5 h
- DDECC POLAR(128,86): about 13 h
- SB-ECC POLAR(128,64): about 7 h
- SB-ECC CCSDS(128,64): about 7 h
- CrossMPT POLAR(128,64): about 6 h

The notebook prints one line every 10 epochs. After a disconnect, run step 1 and this step: each run continues from its last checkpoint (at most 50 epochs are lost).

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

from IPython.display import clear_output
ensure_env()
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
S = check(verbose=False)
todo = [(m, c) for (m, c), s in S.items() if s["train"]["state"] != "trained"]
print(f"{len(todo)} run(s) to train:", ", ".join(f"{m} {name(c)}" for m, c in todo))
for i, (method, c) in enumerate(todo, 1):
    tr = S[(method, c)]["train"]; d = tr["dir"]; d.mkdir(parents=True, exist_ok=True)
    E = METHODS[method]["epochs"]
    cmd = [sys.executable, "-u", METHODS[method]["script"]] + train_args(method, c) + \
          [f"--out_dir={d}", f"--save_every={SAVE_EVERY}", "--no_test"]
    if tr["state"] == "resume":
        cmd += [f"--resume={d / 'last_model'}", f"--start_epoch={tr['epoch'] + 1}"]
    clear_output(wait=True)
    print(f"[{i}/{len(todo)}] {method} {name(c)}  epoch {tr['epoch'] + 1} → {E}  seed {SEED}\nlog: {d}/logging.txt\n" + " ".join(cmd[2:]), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPOS / method, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        tail = (tail + [line])[-30:]
        mm = re.match(r"Training epoch (\d+), Batch (\d+)/(\d+)", line)
        if mm:
            if mm[2] == mm[3] and int(mm[1]) % 10 == 0:
                print(time.strftime("[%m-%d %H:%M] ") + line.strip(), flush=True)
        elif line.strip() and not line.startswith(("Epoch ", "Model Saved", " ", "ECC_Transformer(", "DDECCT(", "SB_ECC(", ")")):
            print(line, end="", flush=True)
    proc.wait()
    if proc.returncode == 0 and read_int(d / "last_epoch.txt") == E:
        (d / "train_done.txt").write_text(str(E))
        print(f"✅ {method} {name(c)} trained", flush=True)
    else:
        print(f"❌ {method} {name(c)} stopped (exit code {proc.returncode}); last output:\n" + "".join(tail))
        break
S = check()

## 5. Test (GPU)

For every trained run and every decoding (DDECC: `reg` and `LS`; SB-ECC: Euler, 10 steps; CrossMPT: one forward pass), at 4, 5 and 6 dB: random codewords → AWGN → the method's own decoder.
- **Model**: DDECC and SB-ECC use `best_model`, as their scripts' final test does. CrossMPT uses the model after the last epoch, as its script tests that one.
- **Stopping rule**: the papers' rule, > 500 frame errors (and > 1e5 frames), or `MAX_FRAMES`.
- **Saving**: progress is saved every 2 minutes to `retrain_v4/tests/`. After a disconnect, run step 1 and this step; it continues from the saved counts.

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

ensure_env()
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
S = check(verbose=False)
for (method, c), s in S.items():
    if s["train"]["state"] != "trained":
        print(f"skip {method} {name(c)}: not trained yet"); continue
    for mode in METHODS[method]["modes"]:
        if all(s["res"][(mode, e)] and s["res"][(mode, e)]["done"] for e in EBNO):
            continue
        print(f"\n===== {method} {name(c)} [{mode}]", flush=True)
        cmd = [sys.executable, "-u", "/content/eval_job.py", f"--method={method}", f"--repo={REPOS / method}",
               f"--code_type={c[0]}", f"--n={c[1]}", f"--k={c[2]}", f"--model={s['train']['model']}",
               f"--state={test_file(method, c, mode)}", f"--mode={mode}", f"--seed={SEED}", f"--batch={TEST_BATCH}",
               f"--min_fer={MIN_FER}", f"--min_frames={MIN_FRAMES}", f"--max_frames={MAX_FRAMES_BY_METHOD[method]}",
               f"--save_every_s={SAVE_EVERY_S}", "--ebno=" + ",".join(map(str, EBNO))]
        r = subprocess.run(cmd)
        if r.returncode != 0:
            raise SystemExit(f"test stopped (exit code {r.returncode}); run step 1 and this step again to continue")
S = check()

## 6. Results: new run vs. paper vs. earlier reproduction (no GPU)

For each run, −ln(BER) at 4 / 5 / 6 dB of:
- the **new run** (this notebook)
- the **paper**: DDECC Table 1 (LS) and Table 1 − Δ of Appendix E (regular), N = 6, d = 128; SB-ECC Table 1; CrossMPT Table 1
- the **earlier reproduction** (your earlier runs with seed 42)

It prints the differences to the paper and draws one panel per run and decoding. Saved to `retrain_v4/figures/` (`results.csv`, `retrain_vs_paper.png`).

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

import matplotlib.pyplot as plt
import matplotlib as mpl

# (method, code, mode) -> [4, 5, 6 dB]
PAPER = {("DDECC", ("BCH", 63, 36), "LS"): [5.11, 7.09, 9.82], ("DDECC", ("BCH", 63, 36), "reg"): [5.22, 7.11, 9.78],
         ("DDECC", ("POLAR", 128, 86), "LS"): [7.60, 10.81, 15.17], ("DDECC", ("POLAR", 128, 86), "reg"): [7.92, 11.02, 15.57],
         ("SB-ECC", ("POLAR", 128, 64), "euler"): [9.03, 13.13, 16.94], ("SB-ECC", ("CCSDS", 128, 64), "euler"): [8.60, 13.43, 19.20],
         ("CrossMPT", ("POLAR", 128, 64), "direct"): [7.52, 11.21, 14.76]}
EARLIER = {("DDECC", ("BCH", 63, 36), "LS"): [4.70, 6.39, 8.86], ("DDECC", ("BCH", 63, 36), "reg"): [4.71, 6.46, 8.94],
           ("DDECC", ("POLAR", 128, 86), "LS"): [7.19, 10.28, 14.15], ("DDECC", ("POLAR", 128, 86), "reg"): [7.44, 10.63, 14.56],
           ("SB-ECC", ("POLAR", 128, 64), "euler"): [8.06, 11.68, 15.98], ("SB-ECC", ("CCSDS", 128, 64), "euler"): [7.57, 11.69, 16.72],
           ("CrossMPT", ("POLAR", 128, 64), "direct"): [5.59, 8.72, 13.0]}

S = check(verbose=False)
rows, panels = ["method,code,decoding,ebno,neg_ln_ber,ber,fer,frames,frame_errors,finished,paper,earlier"], []
print(f"{'run':<34}{'new (4/5/6 dB)':>24}{'paper':>22}{'new − paper':>24}{'earlier − paper':>24}")
for (method, c), s in S.items():
    for mode in METHODS[method]["modes"]:
        key = (method, c, mode)
        new = [s["res"][(mode, e)] for e in EBNO]
        nv = [lnb(r) if r else None for r in new]
        P, Ea = PAPER[key], EARLIER[key]
        f = lambda v: " / ".join("—" if x is None else f"{x:.2f}" for x in v)
        d = [None if x is None else x - p for x, p in zip(nv, P)]
        de = [x - p for x, p in zip(Ea, P)]
        star = "" if all(r and r["done"] for r in new) else " (*unfinished)"
        print(f"{method + ' ' + name(c) + ' ' + mode:<34}{f(nv) + star:>24}{f(P):>22}"
              f"{' / '.join('—' if x is None else f'{x:+.2f}' for x in d):>24}{' / '.join(f'{x:+.2f}' for x in de):>24}")
        for e, r, p, ea in zip(EBNO, new, P, Ea):
            if r:
                rows.append(f"{method},{name(c)},{mode},{e},{lnb(r):.4f},{r['ber']:.6e},{r['fer']:.6e},{r['frames']},{r['fe']},{r['done']},{p},{ea}")
        panels.append((f"{method} {name(c)} [{mode}]", nv, P, Ea))
(FIGS / "results.csv").write_text("\n".join(rows) + "\n")

INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
mpl.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "legend.frameon": False})
fig, axes = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 3.6), constrained_layout=True, squeeze=False)
for ax, (title, nv, P, Ea) in zip(axes[0], panels):
    ax.plot(EBNO, P, color="#eb6834", ls="--", marker="s", lw=2, ms=6, label="paper")
    ax.plot(EBNO, Ea, color="#52514e", ls=":", marker="^", lw=2, ms=6, label="earlier reproduction (seed 42)")
    pts = [(e, v) for e, v in zip(EBNO, nv) if v is not None and math.isfinite(v)]
    if pts:
        ax.plot([e for e, _ in pts], [v for _, v in pts], color="#2a78d6", marker="o", lw=2, ms=6, label=f"new run (seed {SEED})")
    ax.set_title(title, fontsize=9, loc="left", color=INK, fontweight="bold")
    ax.set_xticks(EBNO); ax.set_xlim(3.8, 6.2); ax.set_xlabel("Eb/N0 (dB)")
axes[0][0].set_ylabel("−ln(BER)")
h, l = axes[0][0].get_legend_handles_labels()
fig.legend(h, l, loc="upper right", ncol=3, fontsize=9, bbox_to_anchor=(1, 1.12))
fig.savefig(FIGS / "retrain_vs_paper.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved to", FIGS)